# Bi-Directional Spatial-Temporal Mamba (ST-Mamba) — Production Notebook

**Architecture**: Hilbert-curve ordering → Bi-directional Spatial Mamba → Causal Temporal Mamba

**Target hardware**: 2 × NVIDIA T4 (Kaggle, 2 × 15 GB VRAM)

**Dataset**: URANS CFD — ~800,000 cell unstructured mesh, multi-element airfoil, ~100–150 timesteps

---

## ⏱ Expected Training Time

| Scenario | Time |
|----------|------|
| 800 K cells, 80 epochs, 2 × T4 (AMP + grad-ckpt) | ~18–36 hours |
| 800 K cells, 1 × T4 (AMP + grad-ckpt) | ~36–72 hours |
| Mock 10 K cells, 5 epochs, CPU | ~5–15 minutes |

## 🧠 Memory Estimates (2 × T4, batch_size=1)

| Tensor | Shape | Memory |
|--------|-------|--------|
| Input patch sequence | (1, 12500, 64, 5) | ~1.5 GB |
| Spatial Mamba hidden | (1, 12500, 128) | ~0.6 GB |
| Temporal Mamba hidden | (1, 20, 128) | ~10 MB |
| Model parameters | — | ~200 MB |
| Grad checkpointing savings | — | ~40% VRAM |

> **Kaggle 12-hour session**: Checkpoints are saved every epoch.  
> On restart, the notebook **auto-detects and resumes** from the latest checkpoint.

## 1 — Environment Setup & Installation

In [ ]:
import subprocess, sys, importlib

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Install mamba-ssm only if CUDA is available (Kaggle GPU kernels)
try:
    import torch
    if torch.cuda.is_available():
        try:
            import mamba_ssm  # noqa: F401
            print("mamba_ssm already installed.")
        except ImportError:
            print("Installing mamba-ssm …")
            _pip("mamba-ssm", "causal-conv1d")
    else:
        print("No CUDA detected — mamba_ssm will NOT be installed; GRU fallback will be used.")
except Exception as e:
    print(f"Could not check CUDA: {e}")

# Standard deps (already on Kaggle)
for pkg in ["numpy", "pandas", "scipy", "matplotlib", "scikit-learn", "tqdm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        _pip(pkg)

print("Environment ready.")

### Imports

In [ ]:
import os, math, time, warnings, pathlib, copy, gc
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torch.utils.checkpoint as grad_ckpt

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB")

## 2 — Configuration

In [ ]:
@dataclass
class Config:
    # ── Data ───────────────────────────────────────────────────────────────────
    pressure_path:   str = "/kaggle/input/master-2sec-updated/master_pressure_2sec.csv"
    u_vel_path:      str = "/kaggle/input/master-2sec-updated/master_u_velocity_2sec.csv"
    v_vel_path:      str = "/kaggle/input/master-2sec-updated/master_v_velocity_2sec.csv"
    output_dir:      str = "outputs"
    checkpoint_dir:  str = "checkpoints"

    # ── Mesh / patch ───────────────────────────────────────────────────────────
    n_nodes:         int   = 800_000
    patch_size:      int   = 32       # Reduced from 64 for finer spatial resolution
    n_channels:      int   = 5        # p, u, v, x_coord, y_coord

    # ── Temporal window ────────────────────────────────────────────────────────
    time_in:         int   = 20       # input timesteps
    time_out:        int   = 10       # prediction horizon: model is trained to predict 10 steps at once
                                      #   (dataset yields 10-step targets; eliminates autoregressive error accumulation)
    rollout_steps:   int   = 10       # autoregressive rollout length at inference

    # ── Train / val / test split ───────────────────────────────────────────────
    train_ratio:     float = 0.70
    val_ratio:       float = 0.15     # test = 1 - train_ratio - val_ratio

    # ── Model ──────────────────────────────────────────────────────────────────
    hidden_dim:      int   = 128
    spatial_layers:  int   = 4        # Increased from 2 to capture multi-scale dynamics
    temporal_layers: int   = 4        # Increased from 2 to capture multi-scale dynamics
    spatial_refinement_layers: int = 3    # iterative bidirectional spatial refinement passes after ST blocks
    dropout:         float = 0.15
    mamba_d_state:   int   = 16
    mamba_d_conv:    int   = 4
    mamba_expand:    int   = 2

    # ── Training ───────────────────────────────────────────────────────────────
    epochs:          int   = 80
    patience:        int   = 10
    batch_size:      int   = 1
    lr:              float = 1e-4
    weight_decay:    float = 0.01
    warmup_steps:    int   = 500
    max_steps:       int   = 15_000
    grad_clip:       float = 0.5
    use_amp:         bool  = True
    use_checkpointing: bool = True    # gradient checkpointing to save VRAM

    # ── Autoencoder pretraining (runs before main training loop) ───────────────
    use_autoencoder_pretrain: bool = True
    autoencoder_latent_dim: int = 64
    autoencoder_epochs: int = 5
    autoencoder_lr: float = 1e-3

    # ── Loss ───────────────────────────────────────────────────────────────────
    spectral_weight: float = 0.10
    energy_weight:   float = 0.05
    gradient_weight: float = 0.05    # Physics-informed spatial gradient penalty weight
    spatial_smooth_weight: float = 0.10  # Spatial smoothness regularization weight (λ): penalizes jumps between adjacent patches
    use_spatial_weighting: bool = True   # upweight near-wall cells
    loss_weights: Dict[str, float] = field(
        default_factory=lambda: {"pressure": 0.30, "u_velocity": 0.35, "v_velocity": 0.35}
    )

CONFIG = Config()
os.makedirs(CONFIG.output_dir,     exist_ok=True)
os.makedirs(CONFIG.checkpoint_dir, exist_ok=True)
print("Config initialised.")


## 3 — Hilbert Sorting Module

In [ ]:
def _d2xy(n: int, d: int) -> Tuple[int, int]:
    """Convert Hilbert-curve distance *d* to (x, y) for an n×n grid."""
    x = y = 0
    s = 1
    while s < n:
        rx = 1 if (d & 2) else 0
        ry = 1 if (d & 1) ^ rx else 0
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        x += s * rx
        y += s * ry
        d >>= 2
        s <<= 1
    return x, y


def _xy2d(n: int, x: int, y: int) -> int:
    """Convert (x, y) grid coordinates to Hilbert-curve distance."""
    d = 0
    s = n >> 1
    while s > 0:
        rx = 1 if (x & s) > 0 else 0
        ry = 1 if (y & s) > 0 else 0
        d += s * s * ((3 * rx) ^ ry)
        # rotate
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        s >>= 1
    return d


def hilbert_sort_indices(coords: np.ndarray, order: int = 10) -> np.ndarray:
    """
    Return permutation indices that sort *coords* (N×2) along a 2-D
    Hilbert curve of the given *order* (grid resolution = 2**order).

    Parameters
    ----------
    coords : (N, 2) float array — (x, y) coordinates
    order  : Hilbert curve order; default 10 → 1024×1024 grid

    Returns
    -------
    indices : (N,) int64 array — argsort by Hilbert key
    """
    grid = 1 << order  # 2^order
    # normalise coords to [0, grid-1]
    xy = coords.copy().astype(np.float64)
    for dim in range(2):
        lo, hi = xy[:, dim].min(), xy[:, dim].max()
        rng = hi - lo
        if rng < 1e-12:
            xy[:, dim] = 0.0
        else:
            xy[:, dim] = (xy[:, dim] - lo) / rng * (grid - 1)

    ix = xy[:, 0].astype(np.int64).clip(0, grid - 1)
    iy = xy[:, 1].astype(np.int64).clip(0, grid - 1)

    # Vectorised Hilbert key computation
    keys = np.array([_xy2d(grid, int(ix[i]), int(iy[i])) for i in range(len(ix))],
                    dtype=np.int64)
    return np.argsort(keys)


print("Hilbert module ready.")

## 4 — Data Pipeline

In [ ]:
# ── 4a. Mock data generator ───────────────────────────────────────────────────

def make_mock_data(
    n_nodes: int = 10_000,
    n_timesteps: int = 40,
    seed: int = 0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns
    -------
    coords   : (N, 2) x,y in [0, 1]
    pressure : (N, T)
    u_vel    : (N, T)
    v_vel    : (N, T)
    """
    rng = np.random.default_rng(seed)
    # Scatter points on a unit square, with a denser cluster (mock boundary layer)
    base = rng.uniform(0, 1, (n_nodes, 2)).astype(np.float32)
    n_bl = n_nodes // 10
    base[:n_bl, 1] = rng.uniform(0, 0.05, n_bl)  # near-wall cluster

    coords = base
    t = np.linspace(0, 2 * np.pi, n_timesteps, dtype=np.float32)

    # Synthetic pressure / velocity with spatial + temporal variation
    x, y = coords[:, 0:1], coords[:, 1:2]
    pressure = (
        100_000
        + 5_000 * np.sin(2 * np.pi * x)
        + 3_000 * np.cos(2 * np.pi * y)
        + 1_000 * np.sin(t[np.newaxis, :])
        + rng.normal(0, 100, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)

    u_vel = (
        20
        + 10 * np.cos(np.pi * y)
        + 2 * np.sin(t[np.newaxis, :])
        + rng.normal(0, 0.5, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)

    v_vel = (
        5 * np.sin(np.pi * x)
        + 1 * np.cos(t[np.newaxis, :])
        + rng.normal(0, 0.3, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)

    print(f"[Mock] Generated {n_nodes:,} nodes × {n_timesteps} timesteps.")
    return coords, pressure, u_vel, v_vel


In [ ]:
# ── 4b. Real CSV loader ───────────────────────────────────────────────────────

def load_csv_data(
    pressure_path: str,
    u_vel_path: str,
    v_vel_path: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load three variable CSVs with layout:
      col0 = x_coord, col1 = y_coord, col2..end = values at t0, t1, …

    Returns
    -------
    coords   : (N, 2)
    pressure : (N, T)
    u_vel    : (N, T)
    v_vel    : (N, T)
    """
    print("Loading CSVs …")
    df_p = pd.read_csv(pressure_path, header=0)
    df_u = pd.read_csv(u_vel_path,    header=0)
    df_v = pd.read_csv(v_vel_path,    header=0)

    coords   = df_p.iloc[:, :2].values.astype(np.float32)
    pressure = df_p.iloc[:, 2:].values.astype(np.float32)
    u_vel    = df_u.iloc[:, 2:].values.astype(np.float32)
    v_vel    = df_v.iloc[:, 2:].values.astype(np.float32)

    T = min(pressure.shape[1], u_vel.shape[1], v_vel.shape[1])
    pressure, u_vel, v_vel = pressure[:, :T], u_vel[:, :T], v_vel[:, :T]

    print(f"Loaded — nodes: {coords.shape[0]:,}, timesteps: {T}")
    return coords, pressure, u_vel, v_vel


def auto_load_data(cfg: Config):
    """Auto-detect real CSVs; fall back to mock data with a warning."""
    real = all(os.path.exists(p) for p in [cfg.pressure_path, cfg.u_vel_path, cfg.v_vel_path])
    if real:
        print("Real CSV files detected — loading production data.")
        return load_csv_data(cfg.pressure_path, cfg.u_vel_path, cfg.v_vel_path)
    warnings.warn(
        "Real CSV files NOT found. Using mock data for testing/debugging. "
        "Set correct paths in Config for production runs.",
        UserWarning,
        stacklevel=2,
    )
    print("[WARN] Falling back to mock data (10 K nodes, 40 timesteps).")
    return make_mock_data(n_nodes=10_000, n_timesteps=40)


print("Data pipeline ready.")

In [ ]:
# ── 4c. Normalisation (fit only on training split) ────────────────────────────

class DataNormalizer:
    """
    Improved normalizer that preserves inter-variable relationships.

    - Velocity components (u, v) share a common scale derived from the overall
      velocity magnitude, preserving flow direction and spatial amplitude coupling.
    - Pressure is normalized independently with its own mean/std.
    - Coordinates normalized to [0, 1].

    This avoids the per-variable StandardScaler approach that suppresses the
    relative pressure-velocity relationships critical for accurate flow reconstruction.
    """

    def __init__(self):
        self.means  = {}
        self.scales = {}
        self.coord_min   = None
        self.coord_scale = None

    def fit(self, pressure_train, u_train, v_train, coords):
        """Fit on training-split arrays (N, T_train)."""
        # Pressure: independent standardization
        self.means["pressure"]  = float(pressure_train.mean())
        p_std = float(pressure_train.std())
        self.scales["pressure"] = p_std if p_std > 1e-12 else 1.0

        # Velocity: shared scale from combined magnitude to preserve directional coupling
        self.means["u_velocity"] = float(u_train.mean())
        self.means["v_velocity"] = float(v_train.mean())
        vel_all   = np.concatenate([u_train.ravel(), v_train.ravel()])
        vel_scale = float(vel_all.std())
        if vel_scale < 1e-12:
            vel_scale = 1.0
        # Both u and v use the same scale so the flow direction/magnitude is preserved
        self.scales["u_velocity"] = vel_scale
        self.scales["v_velocity"] = vel_scale

        # Coords
        self.coord_min = coords.min(axis=0)
        rng = coords.max(axis=0) - coords.min(axis=0)
        rng[rng < 1e-12] = 1.0
        self.coord_scale = rng

        print("  Pressure  : mean={:.4f}, scale={:.4f}".format(
            self.means["pressure"], self.scales["pressure"]))
        print("  U-velocity: mean={:.4f}, scale={:.4f} (shared vel scale)".format(
            self.means["u_velocity"], self.scales["u_velocity"]))
        print("  V-velocity: mean={:.4f}, scale={:.4f} (shared vel scale)".format(
            self.means["v_velocity"], self.scales["v_velocity"]))

    def transform_var(self, arr, name):
        shape = arr.shape
        return ((arr - self.means[name]) / self.scales[name]).reshape(shape)

    def inverse_transform_var(self, arr, name):
        shape = arr.shape
        return (arr * self.scales[name] + self.means[name]).reshape(shape)

    def transform_coords(self, coords):
        return (coords - self.coord_min) / self.coord_scale

    def fit_transform(self, pressure, u_vel, v_vel, coords, train_end_idx):
        """Fit on training timesteps, transform all timesteps. Returns normalised arrays."""
        self.fit(
            pressure[:, :train_end_idx],
            u_vel[:, :train_end_idx],
            v_vel[:, :train_end_idx],
            coords,
        )
        p_n = self.transform_var(pressure, "pressure")
        u_n = self.transform_var(u_vel,    "u_velocity")
        v_n = self.transform_var(v_vel,    "v_velocity")
        c_n = self.transform_coords(coords)
        return p_n, u_n, v_n, c_n


print("Normalizer ready.")


In [ ]:
# ── 4d. Patch Dataset ─────────────────────────────────────────────────────────

class PatchDataset(Dataset):
    """
    Produces (input_seq, target_seq) patch-level windows.

    Each sample:
      x : (time_in,  n_patches, patch_size, n_channels)
      y : (time_out, n_patches, patch_size, 3)          — p, u, v only

    For efficiency the data is stored as (N, T, C) and split into patches
    on-the-fly per __getitem__ call.
    """

    def __init__(
        self,
        data:         np.ndarray,   # (N, T, C)  C = p,u,v,x,y
        hilbert_idx:  np.ndarray,   # (N,) — sorted order
        patch_size:   int,
        time_in:      int,
        time_out:     int,
        t_start:      int,
        t_end:        int,
    ):
        super().__init__()
        self.patch_size  = patch_size
        self.time_in     = time_in
        self.time_out    = time_out

        # Reorder nodes along Hilbert curve then pad to multiple of patch_size
        data_sorted = data[hilbert_idx]  # (N, T, C)
        N, T, C = data_sorted.shape
        n_patches = math.ceil(N / patch_size)
        pad_len   = n_patches * patch_size - N
        if pad_len > 0:
            pad = np.zeros((pad_len, T, C), dtype=data_sorted.dtype)
            data_sorted = np.concatenate([data_sorted, pad], axis=0)

        # Reshape → (n_patches, patch_size, T, C)
        self.patches = data_sorted.reshape(n_patches, patch_size, T, C)
        self.n_patches = n_patches

        # Valid window start indices within [t_start, t_end)
        win = time_in + time_out
        self.starts = list(range(t_start, t_end - win + 1))

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        t0 = self.starts[idx]
        # x: (n_patches, patch_size, time_in, C) → transpose → (time_in, n_patches, patch_size, C)
        x = self.patches[:, :, t0 : t0 + self.time_in, :]       # (P, S, T_in, C)
        y = self.patches[:, :, t0 + self.time_in :
                               t0 + self.time_in + self.time_out, :3]  # (P, S, T_out, 3)

        x = torch.from_numpy(x.transpose(2, 0, 1, 3))  # (T_in, P, S, C)
        y = torch.from_numpy(y.transpose(2, 0, 1, 3))  # (T_out, P, S, 3)
        return x.float(), y.float()


print("PatchDataset ready.")

## 5 — Patch Embedding & Unembedding

In [ ]:
class PatchEmbed(nn.Module):
    """Linear projection from (patch_size × n_channels) → hidden_dim.

    Includes distance-weighted spatial scaling: a learned gate based on each
    patch's normalised (x, y) coordinates (channels 3 and 4) modulates the
    embedding magnitude, preserving spatial gradient structure and helping the
    model distinguish patches by their physical location.
    """

    def __init__(self, patch_size: int, n_channels: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.proj = nn.Linear(patch_size * n_channels, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(dropout)
        # Distance-weighted spatial scaling.
        # Input: 2-d normalised coordinates (x_norm, y_norm) averaged over the
        # patch nodes.  Output: per-dimension sigmoid scale in (0, 1).
        self.dist_scale = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, T, P, S, C)  S=patch_size, C=n_channels
        B, T, P, S, C = x.shape
        # Spatial coordinates are in channels 3 and 4 (x_norm, y_norm),
        # matching n_channels=5 layout: [p, u, v, x_coord, y_coord].
        # Average over nodes within the patch to get a per-patch coordinate.
        coords = x[:, :, :, :, 3:5].mean(dim=3)   # (B, T, P, 2) — [x_norm, y_norm]
        x_flat = x.reshape(B, T, P, S * C)
        h = self.drop(self.norm(self.proj(x_flat)))  # (B, T, P, hidden_dim)
        # Modulate embeddings by learned spatial scale factor.
        scale = self.dist_scale(coords)              # (B, T, P, hidden_dim)
        return h * scale


class PatchUnembed(nn.Module):
    """Project hidden_dim → (patch_size × 3)."""

    def __init__(self, patch_size: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, patch_size * 3)
        self.drop = nn.Dropout(dropout)
        self.patch_size = patch_size

    def forward(self, x):
        # x: (B, T, P, hidden_dim)
        B, T, P, H = x.shape
        x = self.drop(self.proj(x))           # (B, T, P, S*3)
        return x.reshape(B, T, P, self.patch_size, 3)


print("Patch embed/unembed ready.")


## 6 — Bi-Directional Spatial Mamba Block

In [ ]:
# ── Mamba wrapper (falls back to GRU if mamba_ssm not available) ──────────────

try:
    from mamba_ssm import Mamba as _MambaSSM
    _MAMBA_AVAILABLE = True
    print("mamba_ssm backend detected.")
except ImportError:
    _MAMBA_AVAILABLE = False
    print("mamba_ssm NOT available — using bidirectional GRU fallback.")


class MambaLikeBlock(nn.Module):
    """
    Single-direction SSM block.
    Uses mamba_ssm.Mamba when available, otherwise a GRU wrapper
    that approximates the same interface.
    """

    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        if _MAMBA_AVAILABLE:
            self._impl = _MambaSSM(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
            )
        else:
            # GRU wrapper: same input/output shape (B, L, d_model)
            d_inner = d_model * expand
            self._impl = nn.Sequential(
                nn.Linear(d_model, d_inner),
                nn.GELU(),
            )
            self._gru  = nn.GRU(d_inner, d_model, batch_first=True)

        self._use_mamba = _MAMBA_AVAILABLE

    def forward(self, x):
        # x: (B, L, D)
        if self._use_mamba:
            return self._impl(x)
        h = self._impl(x)
        out, _ = self._gru(h)
        return out


class BiDirectionalSpatialMamba(nn.Module):
    """
    Bi-directional Mamba applied along the *spatial* (patch) dimension.

    For each timestep T:
      forward  Mamba on patch sequence 0 → P-1
      backward Mamba on patch sequence P-1 → 0
      combine  → linear projection back to hidden_dim

    Uses gradient checkpointing when requested.
    """

    def __init__(self, hidden_dim: int, d_state: int, d_conv: int, expand: int,
                 dropout: float = 0.0, use_checkpointing: bool = False):
        super().__init__()
        self.fwd = MambaLikeBlock(hidden_dim, d_state, d_conv, expand)
        self.bwd = MambaLikeBlock(hidden_dim, d_state, d_conv, expand)
        self.proj   = nn.Linear(2 * hidden_dim, hidden_dim)
        self.norm   = nn.LayerNorm(hidden_dim)
        self.drop   = nn.Dropout(dropout)
        self.use_ckpt = use_checkpointing

    def _spatial_pass(self, x_t):
        # x_t: (B, P, H)
        fwd_out = self.fwd(x_t)                              # (B, P, H)
        bwd_out = self.bwd(x_t.flip(1)).flip(1)             # (B, P, H)
        cat = torch.cat([fwd_out, bwd_out], dim=-1)         # (B, P, 2H)
        return self.drop(self.proj(cat))                     # (B, P, H)

    def forward(self, x):
        # x: (B, T, P, H)
        B, T, P, H = x.shape
        residual = x

        out_list = []
        for t in range(T):
            x_t = x[:, t]  # (B, P, H)
            if self.use_ckpt and self.training:
                out_t = grad_ckpt.checkpoint(self._spatial_pass, x_t, use_reentrant=False)
            else:
                out_t = self._spatial_pass(x_t)
            out_list.append(out_t)

        out = torch.stack(out_list, dim=1)  # (B, T, P, H)
        return self.norm(out + residual)


print("Bi-directional Spatial Mamba ready.")

## 7 — Temporal Mamba Block

In [ ]:
class CausalTemporalMamba(nn.Module):
    """
    Causal (uni-directional) Mamba applied along the *time* dimension.

    For each patch p:
      causal Mamba on time sequence t=0 → T-1
    Causality is enforced by using standard (forward-only) Mamba.

    Uses gradient checkpointing when requested.
    """

    def __init__(self, hidden_dim: int, d_state: int, d_conv: int, expand: int,
                 dropout: float = 0.0, use_checkpointing: bool = False):
        super().__init__()
        self.mamba  = MambaLikeBlock(hidden_dim, d_state, d_conv, expand)
        self.norm   = nn.LayerNorm(hidden_dim)
        self.drop   = nn.Dropout(dropout)
        self.use_ckpt = use_checkpointing

    def _temporal_pass(self, x_p):
        # x_p: (B, T, H)
        return self.drop(self.mamba(x_p))

    def forward(self, x):
        # x: (B, T, P, H)
        B, T, P, H = x.shape
        residual = x

        # process each patch independently along time
        x_p = x.permute(0, 2, 1, 3).reshape(B * P, T, H)  # (B*P, T, H)

        if self.use_ckpt and self.training:
            out = grad_ckpt.checkpoint(self._temporal_pass, x_p, use_reentrant=False)
        else:
            out = self._temporal_pass(x_p)

        out = out.reshape(B, P, T, H).permute(0, 2, 1, 3)  # (B, T, P, H)
        return self.norm(out + residual)


print("Causal Temporal Mamba ready.")

## 8 — ST Block (Spatial + Temporal)

In [ ]:
class STBlock(nn.Module):
    """One ST-Mamba block: BiDirectional Spatial → Causal Temporal."""

    def __init__(self, hidden_dim: int, d_state: int, d_conv: int, expand: int,
                 dropout: float = 0.0, use_checkpointing: bool = False):
        super().__init__()
        self.spatial  = BiDirectionalSpatialMamba(
            hidden_dim, d_state, d_conv, expand, dropout, use_checkpointing)
        self.temporal = CausalTemporalMamba(
            hidden_dim, d_state, d_conv, expand, dropout, use_checkpointing)

    def forward(self, x):
        # x: (B, T, P, H)
        x = self.spatial(x)
        x = self.temporal(x)
        return x


print("ST Block ready.")

## 9 — Full ST-Mamba Model

In [ ]:
class STMambaModel(nn.Module):
    """
    Full Bi-Directional Spatial-Temporal Mamba model.

    Architecture:
        embed → [ST-Block₁ … ST-Blockₙ]
              → [SpatialRefine₁ … SpatialRefineₘ]   (m = spatial_refinement_layers)
              → time_proj
              → unembed

    The spatial refinement layers are additional BiDirectionalSpatialMamba
    passes that smooth and refine the spatial field after the main ST blocks,
    correcting patch-boundary artifacts and improving spatial coherence.

    Input  : (B, T_in,  n_patches, patch_size, n_channels)
    Output : (B, T_out, n_patches, patch_size, 3)   — p, u, v
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        self.embed   = PatchEmbed(cfg.patch_size, cfg.n_channels, cfg.hidden_dim, cfg.dropout)
        self.blocks  = nn.ModuleList([
            STBlock(
                cfg.hidden_dim,
                cfg.mamba_d_state,
                cfg.mamba_d_conv,
                cfg.mamba_expand,
                cfg.dropout,
                cfg.use_checkpointing,
            )
            for _ in range(cfg.spatial_layers)           # spatial_layers ≡ number of ST blocks
        ])

        # Spatial refinement layers: iterative bidirectional spatial Mamba
        # passes that run after the main ST blocks to smooth patch boundaries
        # and improve spatial field coherence.
        self.spatial_refinement = nn.ModuleList([
            BiDirectionalSpatialMamba(
                cfg.hidden_dim,
                cfg.mamba_d_state,
                cfg.mamba_d_conv,
                cfg.mamba_expand,
                cfg.dropout,
                cfg.use_checkpointing,
            )
            for _ in range(cfg.spatial_refinement_layers)
        ])

        self.unembed = PatchUnembed(cfg.patch_size, cfg.hidden_dim, cfg.dropout)

        # Learned temporal projection: T_in → T_out
        self.time_proj = nn.Linear(cfg.time_in, cfg.time_out)

        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"STMambaModel created — {n_params:,} trainable parameters.")

    def forward(self, x):
        # x: (B, T_in, P, S, C)
        h = self.embed(x)          # (B, T_in, P, H)

        for block in self.blocks:
            h = block(h)           # (B, T_in, P, H)

        # Iterative spatial refinement: smooth patch-boundary artifacts
        for refine in self.spatial_refinement:
            h = refine(h)          # (B, T_in, P, H)

        # Project time dimension: (B, P, H, T_in) → (B, P, H, T_out)
        h = h.permute(0, 2, 3, 1)                         # (B, P, H, T_in)
        h = self.time_proj(h)                              # (B, P, H, T_out)
        h = h.permute(0, 3, 1, 2)                         # (B, T_out, P, H)

        out = self.unembed(h)      # (B, T_out, P, S, 3)
        return out


print("STMambaModel class ready.")


## 10 — Loss Functions

In [ ]:
def compute_wall_weights(
    coords_norm: np.ndarray,   # (N, 2) normalised to [0,1]
    sigma: float = 0.02,
    max_weight: float = 10.0,
) -> torch.Tensor:
    """
    Compute per-node spatial weights based on distance from the
    nearest (presumed) wall.  We approximate wall proximity by finding
    the minimum y-coordinate cluster (boundary layer region).

    Returns a float tensor of shape (N,) with values in [1, max_weight].
    """
    y = coords_norm[:, 1]
    # wall distance ≈ y-normalised (assumes wall at y≈0)
    wall_dist = y.clip(1e-6, None)
    # inverse distance, clamped
    weights = (sigma / wall_dist).clip(1.0, max_weight)
    return torch.from_numpy(weights.astype(np.float32))


class STMambaLoss(nn.Module):
    """
    Combined physics-aware loss:
      L = w_mse      * MSE
        + w_spec     * spectral_loss
        + w_energy   * energy_loss
        + w_gradient * spatial_gradient_loss

    The spatial gradient penalty encourages spatial continuity and momentum
    balance by matching predicted and target gradients across patches.

    All terms are variable-weighted by cfg.loss_weights.
    Optionally applies spatial weighting (upweight near-wall cells).
    """

    def __init__(
        self,
        cfg: Config,
        wall_weights: Optional[torch.Tensor] = None,
        patch_size: Optional[int] = None,
    ):
        super().__init__()
        self.cfg = cfg
        self.var_w = torch.tensor(
            [cfg.loss_weights["pressure"],
             cfg.loss_weights["u_velocity"],
             cfg.loss_weights["v_velocity"]],
            dtype=torch.float32,
        )
        self.register_buffer("var_weights", self.var_w)

        # spatial weights: (n_patches, patch_size) → (n_patches * patch_size,)
        if cfg.use_spatial_weighting and wall_weights is not None:
            self.register_buffer("spatial_weights", wall_weights)  # (N_padded,)
        else:
            self.spatial_weights = None

    def _mse_loss(self, pred, target):
        # pred, target: (B, T, P, S, 3)
        err = (pred - target) ** 2  # (B, T, P, S, 3)

        if self.spatial_weights is not None:
            # spatial_weights: (P*S,) → reshape to (1, 1, P, S, 1)
            P, S = pred.shape[2], pred.shape[3]
            sw = self.spatial_weights[: P * S].reshape(1, 1, P, S, 1).to(pred.device)
            err = err * sw

        # variable-wise weighting
        vw = self.var_weights.to(pred.device).reshape(1, 1, 1, 1, 3)
        err = (err * vw).mean()
        return err

    def _spectral_loss(self, pred, target):
        # Compute FFT along time dimension (dim=1) and match magnitude spectra
        # pred, target: (B, T, P, S, 3)
        pred_fft   = torch.fft.rfft(pred,   dim=1)
        target_fft = torch.fft.rfft(target, dim=1)
        loss = F.mse_loss(pred_fft.abs(), target_fft.abs())
        return loss

    def _energy_loss(self, pred, target):
        # Kinetic energy: 0.5 * (u^2 + v^2), indices 1 and 2
        ke_pred   = 0.5 * (pred[..., 1] ** 2 + pred[..., 2] ** 2)
        ke_target = 0.5 * (target[..., 1] ** 2 + target[..., 2] ** 2)
        return F.mse_loss(ke_pred, ke_target)

    def _gradient_loss(self, pred, target):
        """
        Physics-informed spatial gradient penalty.

        Computes finite differences along the patch dimension (proxy for spatial
        gradients) and penalizes deviations from the ground-truth gradients.
        This encourages spatial continuity and is a soft surrogate for momentum
        balance / pressure-velocity coupling across the domain.

        pred, target: (B, T, P, S, 3)
        """
        if pred.shape[2] < 2:
            return pred.new_tensor(0.0)
        # Finite differences along the patch (spatial) axis
        pred_grad   = pred[:, :, 1:] - pred[:, :, :-1]    # (B, T, P-1, S, 3)
        target_grad = target[:, :, 1:] - target[:, :, :-1]
        return F.mse_loss(pred_grad, target_grad)

    def _spatial_smooth_loss(self, pred):
        """
        Spatial smoothness regularizer (configurable λ = spatial_smooth_weight).

        Penalizes large magnitude jumps between spatially adjacent patches,
        correcting prediction discontinuities across patch boundaries and
        improving spatial field coherence in the reconstructed flow field.

        Patch adjacency assumption: patches are ordered along a Hilbert space-
        filling curve (see hilbert_sort_indices), which guarantees that
        consecutive patches in the sequence are spatially adjacent in the mesh.
        Finite differences along the patch dimension therefore approximate true
        spatial gradients rather than penalizing arbitrary reorderings.

        pred: (B, T, P, S, 3)
        """
        if pred.shape[2] < 2:
            return pred.new_tensor(0.0)
        # Finite differences along the patch dimension (Hilbert-ordered spatial axis)
        jumps = pred[:, :, 1:] - pred[:, :, :-1]   # (B, T, P-1, S, 3)
        return (jumps ** 2).mean()

    def forward(self, pred, target):
        mse = self._mse_loss(pred, target)
        total = mse
        losses = {"mse": mse.item()}

        if self.cfg.spectral_weight > 0:
            spec = self._spectral_loss(pred, target)
            total = total + self.cfg.spectral_weight * spec
            losses["spectral"] = spec.item()

        if self.cfg.energy_weight > 0:
            en = self._energy_loss(pred, target)
            total = total + self.cfg.energy_weight * en
            losses["energy"] = en.item()

        if self.cfg.gradient_weight > 0:
            grad_loss = self._gradient_loss(pred, target)
            total = total + self.cfg.gradient_weight * grad_loss
            losses["gradient"] = grad_loss.item()

        if self.cfg.spatial_smooth_weight > 0:
            smooth = self._spatial_smooth_loss(pred)
            total = total + self.cfg.spatial_smooth_weight * smooth
            losses["spatial_smooth"] = smooth.item()

        losses["total"] = total.item()
        return total, losses


print("Loss functions ready.")


## 11 — Training Loop

In [ ]:
def get_lr_scheduler(optimizer, warmup_steps: int, max_steps: int):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def save_checkpoint(state: dict, epoch: int, cfg: Config, is_best: bool = False):
    path = os.path.join(cfg.checkpoint_dir, f"checkpoint_epoch_{epoch:04d}.pth")
    torch.save(state, path)
    # Keep only the latest 3 epoch checkpoints
    ckpts = sorted(pathlib.Path(cfg.checkpoint_dir).glob("checkpoint_epoch_*.pth"))
    for old in ckpts[:-3]:
        old.unlink()
    # Always keep a 'latest' symlink / copy
    latest = os.path.join(cfg.checkpoint_dir, "latest_checkpoint.pth")
    torch.save(state, latest)
    if is_best:
        best = os.path.join(cfg.checkpoint_dir, "best_checkpoint.pth")
        torch.save(state, best)
        print(f"  ✔ Best checkpoint saved (epoch {epoch})")


def load_checkpoint(cfg: Config, model, optimizer, scheduler, scaler):
    """Load latest checkpoint if available. Returns (start_epoch, best_val_loss, history)."""
    latest = os.path.join(cfg.checkpoint_dir, "latest_checkpoint.pth")
    if not os.path.exists(latest):
        print("No checkpoint found — starting fresh.")
        return 0, float("inf"), {"train_loss": [], "val_loss": [], "val_rmse": []}

    print(f"Checkpoint detected — resuming from {latest}")
    ckpt = torch.load(latest, map_location=DEVICE)
    model_state = ckpt["model_state"]
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(model_state)
    else:
        model.load_state_dict(model_state)

    optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt.get("scheduler_state"):
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if scaler is not None and ckpt.get("scaler_state"):
        scaler.load_state_dict(ckpt["scaler_state"])

    start_epoch   = ckpt["epoch"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    history       = ckpt.get("history", {"train_loss": [], "val_loss": [], "val_rmse": []})
    print(f"  Resumed from epoch {ckpt['epoch']}, best_val_loss={best_val_loss:.6f}")
    return start_epoch, best_val_loss, history


def compute_val_rmse(model, val_loader, normalizer, cfg, device):
    """Compute per-variable RMSE in physical units on the validation set."""
    model.eval()
    sq_sum  = np.zeros(3)
    n_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            with autocast(enabled=cfg.use_amp):
                pred = model(x)
            # pred, y: (B, T_out, P, S, 3)
            err = (pred - y).cpu().numpy()
            # Denormalise per variable using the updated normalizer API
            for v_idx, v_name in enumerate(["pressure", "u_velocity", "v_velocity"]):
                e_v   = err[..., v_idx].reshape(-1)
                e_phy = normalizer.scales[v_name] * e_v
                sq_sum[v_idx] += (e_phy ** 2).sum()
            n_total += err[..., 0].size
    rmse = np.sqrt(sq_sum / max(n_total, 1))
    return rmse   # (3,)



class TemporalInputAutoencoder(nn.Module):
    """Lightweight autoencoder for unsupervised reconstruction of input windows."""

    def __init__(self, n_channels: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_channels, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, n_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T_in, n_patches, patch_size, n_channels)
        flat = x.reshape(-1, x.shape[-1])
        z = self.encoder(flat)
        recon = self.decoder(z)
        return recon.reshape_as(x)


def pretrain_autoencoder(train_loader, cfg: Config, device: torch.device):
    """Train an autoencoder before the supervised ST-Mamba training loop."""
    if not cfg.use_autoencoder_pretrain or cfg.autoencoder_epochs <= 0:
        print("Autoencoder pretraining disabled.")
        return None

    autoencoder = TemporalInputAutoencoder(
        n_channels=cfg.n_channels, latent_dim=cfg.autoencoder_latent_dim
    ).to(device)
    optimizer = torch.optim.AdamW(autoencoder.parameters(), lr=cfg.autoencoder_lr)
    scaler = GradScaler(enabled=cfg.use_amp)

    print(
        f"Starting autoencoder pretraining for {cfg.autoencoder_epochs} epochs "
        f"(latent_dim={cfg.autoencoder_latent_dim}) ..."
    )

    for epoch in range(cfg.autoencoder_epochs):
        autoencoder.train()
        epoch_losses = []

        for x, _ in tqdm(train_loader, desc=f"AE Epoch {epoch+1}/{cfg.autoencoder_epochs}", leave=False):
            x = x.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=cfg.use_amp):
                recon = autoencoder(x)
                loss = F.mse_loss(recon, x)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(autoencoder.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            epoch_losses.append(loss.item())

        print(f"  AE epoch {epoch+1:2d}/{cfg.autoencoder_epochs} | recon_loss={np.mean(epoch_losses):.6f}")

    print("Autoencoder pretraining complete.")
    return autoencoder

def train(model, train_loader, val_loader, criterion, normalizer, cfg, device):
    """Main training loop with validation, early stopping, and checkpoint save/resume."""

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                                  weight_decay=cfg.weight_decay)
    total_steps = cfg.epochs * len(train_loader)
    scheduler   = get_lr_scheduler(optimizer, cfg.warmup_steps,
                                   min(cfg.max_steps, total_steps))
    scaler = GradScaler(enabled=cfg.use_amp)

    start_epoch, best_val_loss, history = load_checkpoint(
        cfg, model, optimizer, scheduler, scaler)

    patience_counter = 0
    global_step      = start_epoch * len(train_loader)

    for epoch in range(start_epoch, cfg.epochs):
        epoch_start = time.time()
        model.train()
        train_losses = []

        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=cfg.use_amp):
                pred = model(x)
                loss, loss_dict = criterion(pred, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            global_step += 1
            train_losses.append(loss_dict["total"])

        avg_train = float(np.mean(train_losses))
        history["train_loss"].append(avg_train)

        # ── Validation ────────────────────────────────────────────────────────
        val_loss = float("nan")
        val_rmse = [float("nan")] * 3
        if len(val_loader) > 0:
            model.eval()
            vl = []
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    with autocast(enabled=cfg.use_amp):
                        pred = model(x)
                        l, _ = criterion(pred, y)
                    vl.append(l.item())
            val_loss = float(np.mean(vl))
            val_rmse = compute_val_rmse(model, val_loader, normalizer, cfg, device).tolist()

        history["val_loss"].append(val_loss)
        history["val_rmse"].append(val_rmse)

        elapsed = time.time() - epoch_start
        print(
            f"Epoch {epoch+1:3d}/{cfg.epochs} | "
            f"train={avg_train:.4f} | val={val_loss:.4f} | "
            f"RMSE P={val_rmse[0]:.2f} U={val_rmse[1]:.2f} V={val_rmse[2]:.2f} | "
            f"{elapsed:.1f}s"
        )

        # ── Save checkpoint ────────────────────────────────────────────────────
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        is_best   = not math.isnan(val_loss) and val_loss < best_val_loss
        if is_best:
            best_val_loss    = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        state = {
            "epoch":          epoch,
            "model_state":    raw_model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state":    scaler.state_dict(),
            "best_val_loss":  best_val_loss,
            "history":        history,
        }
        save_checkpoint(state, epoch, cfg, is_best=is_best)

        # ── Early stopping ─────────────────────────────────────────────────────
        if patience_counter >= cfg.patience:
            print(f"Early stopping after {cfg.patience} epochs without improvement.")
            break

    print("Training complete.")
    return history


print("Training loop ready.")

## 12 — Direct Multi-Step Prediction

In [ ]:
@torch.no_grad()
def autoregressive_rollout(
    model,
    initial_window: torch.Tensor,   # (1, T_in, P, S, C)
    rollout_steps:  int,
    device:         torch.device,
    use_amp:        bool = True,
) -> torch.Tensor:
    """
    Multi-step forecasting rollout.

    When the model is configured with T_out >= rollout_steps (direct multi-step
    mode) a single forward pass produces all requested predictions without any
    autoregressive feedback — eliminating error accumulation entirely.

    When T_out < rollout_steps the function falls back to a chunked rollout,
    advancing the sliding window by T_out steps at a time and only feeding
    predictions back when the prediction horizon is exhausted.

    Parameters
    ----------
    initial_window : (1, T_in, P, S, C) — the seed window
    rollout_steps  : number of future timesteps to predict

    Returns
    -------
    predictions : (rollout_steps, P, S, 3)  — p, u, v predictions
    """
    model.eval()
    window = initial_window.to(device)   # (1, T_in, P, S, C)
    C      = window.shape[-1]

    # Determine how many steps the model predicts per forward pass
    t_out = model.cfg.time_out if hasattr(model, "cfg") else 1

    # ── Direct multi-step mode (no autoregressive feedback) ──────────────────
    if t_out >= rollout_steps:
        with autocast(enabled=use_amp):
            pred = model(window)           # (1, T_out, P, S, 3)
        # Trim to exactly rollout_steps and remove batch dimension
        return pred[0, :rollout_steps].cpu()   # (rollout_steps, P, S, 3)

    # ── Chunked rollout (t_out < rollout_steps) ───────────────────────────────
    all_preds = []
    steps_done = 0

    # Reference kinetic energy from the input window (mean over last few steps).
    # Used for dampening-correction to prevent amplitude decay during rollout.
    ref_ke = 0.5 * (window[..., 1] ** 2 + window[..., 2] ** 2).mean().item()

    while steps_done < rollout_steps:
        with autocast(enabled=use_amp):
            pred = model(window)              # (1, t_out, P, S, 3)

        steps_to_take = min(t_out, rollout_steps - steps_done)

        for s in range(steps_to_take):
            pred_step = pred[:, s:s+1]        # (1, 1, P, S, 3)

            # ── Dampening-correction ────────────────────────────────────────
            # When predicted kinetic energy drifts below the reference level we
            # rescale velocity channels to counteract amplitude decay.
            pred_ke = 0.5 * (pred_step[..., 1] ** 2 + pred_step[..., 2] ** 2).mean().item()
            if ref_ke > 1e-8 and pred_ke > 1e-8:
                correction = math.sqrt(ref_ke / pred_ke)
                correction = max(0.5, min(correction, 1.5))
                pred_step = pred_step.clone()
                pred_step[..., 1:3] = pred_step[..., 1:3] * correction
            # ────────────────────────────────────────────────────────────────

            all_preds.append(pred_step[0, 0].cpu())   # (P, S, 3)

        steps_done += steps_to_take

        # Slide the window forward by t_out steps using the new predictions
        new_frames = pred[:, :steps_to_take].clone()   # (1, steps_to_take, P, S, 3)
        # Pad to full C channels (preserve x,y coordinates from last window frame)
        pad = window[:, -steps_to_take:, :, :, 3:].clone()   # (1, steps_to_take, P, S, C-3)
        new_frames_padded = torch.cat([new_frames, pad], dim=-1)  # (1, steps_to_take, P, S, C)
        window = torch.cat([window[:, steps_to_take:], new_frames_padded], dim=1)

    return torch.stack(all_preds, dim=0)   # (rollout_steps, P, S, 3)


print("Autoregressive rollout ready.")


## 13 — CSV Export

In [ ]:
def export_predictions_to_csv(
    predictions:   np.ndarray,    # (time_out, N_padded, 3) — p,u,v normalised
    coords_orig:   np.ndarray,    # (N_real, 2)  — original (x, y)
    normalizer:    DataNormalizer,
    cfg:           Config,
    prefix:        str = "st_mamba_pred",
):
    """
    Denormalise and write three CSV files matching input format.

    Output columns: x_coord, y_coord, pred_t1, pred_t2, …
    Predictions are cropped to real mesh size (removes padding).
    """
    os.makedirs(cfg.output_dir, exist_ok=True)
    N_real   = len(coords_orig)
    R, N_pad, _ = predictions.shape
    N        = min(N_real, N_pad)

    var_names = ["pressure", "u_velocity", "v_velocity"]
    file_stems = [
        f"{prefix}_pressure",
        f"{prefix}_u_velocity",
        f"{prefix}_v_velocity",
    ]

    for v_idx, (v_name, stem) in enumerate(zip(var_names, file_stems)):
        # predictions: (R, N_pad, 3) → pick variable v_idx → (R, N)
        pred_v = predictions[:, :N, v_idx]               # (R, N)
        pred_v = normalizer.inverse_transform_var(pred_v.T, v_name)  # (N, R)

        df = pd.DataFrame(coords_orig[:N], columns=["x_coord", "y_coord"])
        for step in range(R):
            df[f"pred_t{step+1}"] = pred_v[:, step]

        out_path = os.path.join(cfg.output_dir, f"{stem}.csv")
        df.to_csv(out_path, index=False)
        print(f"Exported: {out_path}  ({N:,} rows, {R} pred timesteps)")


print("CSV export ready.")

In [ ]:
# ── Extended validation metrics ───────────────────────────────────────────────

def compute_energy_conservation_error(
    pred_flat: np.ndarray,   # (R, N, 3)  — rollout predictions in normalised space
    normalizer: "DataNormalizer",
) -> float:
    """
    Compute the relative energy conservation error across rollout steps.
    Returns the mean fractional deviation in kinetic energy per step.
    """
    u_phys = normalizer.inverse_transform_var(pred_flat[..., 1], "u_velocity")
    v_phys = normalizer.inverse_transform_var(pred_flat[..., 2], "v_velocity")
    ke = 0.5 * (u_phys ** 2 + v_phys ** 2)              # (R, N)
    ke_mean = ke.mean(axis=1)                             # (R,)
    if ke_mean[0] < 1e-12:
        return float("nan")
    # fractional change from step to step
    diffs = np.abs(np.diff(ke_mean)) / (ke_mean[:-1] + 1e-12)
    return float(diffs.mean())


def compute_gradient_fidelity(
    pred_flat: np.ndarray,    # (R, N, 3)
    truth_flat: np.ndarray,   # (R, N, 3)
) -> float:
    """
    Gradient fidelity: compares spatial finite differences of pressure
    in predicted vs ground-truth fields. Returns the mean relative L2 error.
    """
    errors = []
    for r in range(min(len(pred_flat), len(truth_flat))):
        dp_pred  = np.diff(pred_flat[r, :, 0])
        dp_truth = np.diff(truth_flat[r, :, 0])
        denom = np.linalg.norm(dp_truth) + 1e-12
        errors.append(np.linalg.norm(dp_pred - dp_truth) / denom)
    return float(np.mean(errors)) if errors else float("nan")


def compute_spectral_correlation(
    pred_flat: np.ndarray,    # (R, N, 3)
    truth_flat: np.ndarray,   # (R, N, 3)
    var_idx: int = 1,         # 0=p, 1=u, 2=v
) -> float:
    """
    Spectral correlation: Pearson correlation between power spectra
    (over nodes) averaged across rollout steps. A value close to 1
    indicates the model captures the correct frequency content.
    """
    correlations = []
    for r in range(min(len(pred_flat), len(truth_flat))):
        p_spec = np.abs(np.fft.rfft(pred_flat[r, :, var_idx])) ** 2
        t_spec = np.abs(np.fft.rfft(truth_flat[r, :, var_idx])) ** 2
        if p_spec.std() < 1e-12 or t_spec.std() < 1e-12:
            continue
        corr = np.corrcoef(p_spec, t_spec)[0, 1]
        correlations.append(corr)
    return float(np.mean(correlations)) if correlations else float("nan")


def report_extended_metrics(
    pred_flat: np.ndarray,    # (R, N, 3)  normalised
    truth_flat: np.ndarray,   # (R, N, 3)  normalised
    normalizer: "DataNormalizer",
) -> None:
    """Print all extended validation metrics."""
    ece  = compute_energy_conservation_error(pred_flat, normalizer)
    gf   = compute_gradient_fidelity(pred_flat, truth_flat)
    sc_u = compute_spectral_correlation(pred_flat, truth_flat, var_idx=1)
    sc_v = compute_spectral_correlation(pred_flat, truth_flat, var_idx=2)
    print("── Extended Validation Metrics ─────────────────────────────────────")
    print(f"  Energy conservation error (mean ΔKE/KE) : {ece:.4f}")
    print(f"  Gradient fidelity (∇p L2 error)          : {gf:.4f}")
    print(f"  Spectral correlation — u                 : {sc_u:.4f}")
    print(f"  Spectral correlation — v                 : {sc_v:.4f}")
    print("────────────────────────────────────────────────────────────────────")


print("Extended validation metrics ready.")


## 14 — Visualization

In [ ]:
def plot_training_history(history: dict, cfg: Config):
    """Plot training/validation loss and per-variable RMSE."""
    epochs_ran = len(history["train_loss"])
    if epochs_ran == 0:
        print("No training history to plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Loss curves ────────────────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(history["train_loss"], label="Train Loss")
    if any(not math.isnan(v) for v in history["val_loss"]):
        ax.plot(history["val_loss"], label="Val Loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Loss Curves"); ax.legend(); ax.grid(True)

    # ── Per-variable RMSE ──────────────────────────────────────────────────────
    ax = axes[1]
    rmse_arr = np.array(history["val_rmse"])   # (epochs, 3)
    if rmse_arr.ndim == 2 and rmse_arr.shape[1] == 3:
        for i, name in enumerate(["Pressure", "U-vel", "V-vel"]):
            valid = ~np.isnan(rmse_arr[:, i])
            if valid.any():
                ax.plot(np.where(valid)[0], rmse_arr[valid, i], label=name)
    ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE (physical units)")
    ax.set_title("Validation RMSE per Variable"); ax.legend(); ax.grid(True)

    plt.tight_layout()
    out = os.path.join(cfg.output_dir, "training_history.png")
    plt.savefig(out, dpi=120)
    plt.show()
    print(f"Saved: {out}")


def plot_prediction_samples(
    pred_rollout: np.ndarray,   # (R, N_padded, 3)
    target_data:  np.ndarray,   # (N, T) per variable — just for reference
    coords:       np.ndarray,   # (N, 2)
    normalizer:   DataNormalizer,
    cfg:          Config,
    n_sample_nodes: int = 5,
    seed: int = 42,
):
    """
    Plot ground-truth vs prediction time-series for a few random nodes.
    One figure per variable (P, U, V).
    """
    N_real = len(coords)
    N_pred = min(N_real, pred_rollout.shape[1])
    rng    = np.random.default_rng(seed)
    nodes  = rng.choice(N_pred, size=min(n_sample_nodes, N_pred), replace=False)

    var_names = ["pressure", "u_velocity", "v_velocity"]
    var_labels = ["Pressure", "U-velocity", "V-velocity"]

    for v_idx, (v_name, v_label) in enumerate(zip(var_names, var_labels)):
        fig, axes = plt.subplots(len(nodes), 1, figsize=(12, 3 * len(nodes)), squeeze=False)
        for row, node in enumerate(nodes):
            # pred_rollout: (R, N_padded, 3) → pick node → (R,)
            pred_phys = normalizer.inverse_transform_var(
                pred_rollout[:, node, v_idx].reshape(-1, 1), v_name
            ).flatten()
            axes[row, 0].plot(pred_phys, label="Prediction", color="tab:orange")
            axes[row, 0].set_title(f"Node {node}  ({v_label})")
            axes[row, 0].legend(loc="upper right")
            axes[row, 0].grid(True)

        plt.suptitle(f"Autoregressive Rollout — {v_label}", y=1.01)
        plt.tight_layout()
        out = os.path.join(cfg.output_dir, f"rollout_{v_name}.png")
        plt.savefig(out, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")


print("Visualization ready.")

## 15 — Main Pipeline

In [ ]:
def build_dataloaders(cfg: Config):
    """
    Full data pipeline:
      load → Hilbert sort → normalise → split → PatchDataset → DataLoader

    Returns
    -------
    train_loader, val_loader, test_loader, normalizer,
    coords_orig, pressure, u_vel, v_vel,
    hilbert_idx, n_patches
    """
    # 1. Load data
    coords, pressure, u_vel, v_vel = auto_load_data(cfg)
    N, T = pressure.shape
    print(f"Dataset — nodes: {N:,}, timesteps: {T}")

    # 2. Temporal split indices
    T_train = int(T * cfg.train_ratio)
    T_val   = int(T * cfg.val_ratio)
    T_test  = T - T_train - T_val
    print(f"Temporal split — train: [0, {T_train}), "
          f"val: [{T_train}, {T_train+T_val}), "
          f"test: [{T_train+T_val}, {T})")

    # 3. Normalise (fit on training data only)
    normalizer = DataNormalizer()
    p_n, u_n, v_n, c_n = normalizer.fit_transform(
        pressure, u_vel, v_vel, coords, T_train)

    # 4. Hilbert sort
    print("Computing Hilbert sort …")
    hilbert_idx = hilbert_sort_indices(coords)

    # 5. Stack into (N, T, C): p, u, v, x_norm, y_norm
    x_rep = np.repeat(c_n[:, 0:1], T, axis=1)   # (N, T)
    y_rep = np.repeat(c_n[:, 1:2], T, axis=1)   # (N, T)
    data  = np.stack([p_n, u_n, v_n, x_rep, y_rep], axis=2).astype(np.float32)  # (N, T, 5)

    # 6. Wall weights (for loss)
    wall_weights = None
    if cfg.use_spatial_weighting:
        n_patches  = math.ceil(N / cfg.patch_size)
        pad_len    = n_patches * cfg.patch_size - N
        ww         = compute_wall_weights(c_n)                        # (N,)
        if pad_len > 0:
            ww = torch.cat([ww, torch.ones(pad_len)])                 # (N_padded,)
        # reorder along Hilbert curve (same as data)
        ww_sorted = ww[hilbert_idx]
        if pad_len > 0:
            ww_sorted = torch.cat([ww_sorted, torch.ones(pad_len)])
        wall_weights = ww_sorted

    # 7. Datasets
    n_patches = math.ceil(N / cfg.patch_size)

    def make_ds(t_start, t_end):
        return PatchDataset(data, hilbert_idx, cfg.patch_size,
                            cfg.time_in, cfg.time_out, t_start, t_end)

    ds_train = make_ds(0,              T_train)
    ds_val   = make_ds(T_train,        T_train + T_val)
    ds_test  = make_ds(T_train + T_val, T)

    print(f"Dataset sizes — train: {len(ds_train):,}, "
          f"val: {len(ds_val):,}, test: {len(ds_test):,}")
    if len(ds_train) == 0:
        raise RuntimeError(
            "Training dataset is empty! Increase number of timesteps or reduce "
            "time_in + time_out relative to the training split size."
        )

    # 8. DataLoaders
    dl_kwargs = dict(batch_size=cfg.batch_size, num_workers=0, pin_memory=DEVICE.type == "cuda")
    train_loader = DataLoader(ds_train, shuffle=True,  **dl_kwargs)
    val_loader   = DataLoader(ds_val,   shuffle=False, **dl_kwargs)
    test_loader  = DataLoader(ds_test,  shuffle=False, **dl_kwargs)

    return (train_loader, val_loader, test_loader,
            normalizer, coords, pressure, u_vel, v_vel,
            hilbert_idx, n_patches, wall_weights)


def create_model(cfg: Config, n_patches: int) -> nn.Module:
    model = STMambaModel(cfg)
    n_gpus = torch.cuda.device_count()
    if n_gpus > 1:
        model = nn.DataParallel(model)
        print(f"Using DataParallel across {n_gpus} GPUs.")
    return model.to(DEVICE)


print("Main pipeline helpers ready.")

## 16 — Run

In [ ]:
def main(cfg: Config = CONFIG):
    print("=" * 70)
    print("  ST-Mamba Production Pipeline")
    print("=" * 70)

    # ── Build data ────────────────────────────────────────────────────────────
    (train_loader, val_loader, test_loader,
     normalizer, coords, pressure, u_vel, v_vel,
     hilbert_idx, n_patches, wall_weights) = build_dataloaders(cfg)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = create_model(cfg, n_patches)

    # ── Loss ──────────────────────────────────────────────────────────────────
    criterion = STMambaLoss(cfg, wall_weights=wall_weights, patch_size=cfg.patch_size)
    criterion = criterion.to(DEVICE)

    # ── Pretrain autoencoder (optional) ───────────────────────────────────────
    _ = pretrain_autoencoder(train_loader, cfg, DEVICE)

    # ── Train ─────────────────────────────────────────────────────────────────
    history = train(model, train_loader, val_loader, criterion, normalizer, cfg, DEVICE)

    # ── Visualise training ────────────────────────────────────────────────────
    plot_training_history(history, cfg)

    # ── Load best model for inference ─────────────────────────────────────────
    best_ckpt = os.path.join(cfg.checkpoint_dir, "best_checkpoint.pth")
    if os.path.exists(best_ckpt):
        ckpt_data = torch.load(best_ckpt, map_location=DEVICE)
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        raw_model.load_state_dict(ckpt_data["model_state"])
        print("Loaded best model checkpoint for inference.")

    # ── Direct multi-step prediction on test set ─────────────────────────────
    print(f"\nRunning direct multi-step prediction ({cfg.time_out} steps) …")
    if len(test_loader) > 0:
        x_seed, _ = next(iter(test_loader))   # (B, T_in, P, S, C)
    else:
        # Fall back to last training batch
        x_seed, _ = next(iter(train_loader))
    x_seed = x_seed[:1]  # single sample

    predictions = direct_batch_prediction(
        model, x_seed, DEVICE, cfg.use_amp
    )  # (time_out, P, S, 3)

    # Flatten patches: (R, P*S, 3)
    R, P, S, Cv = predictions.shape
    pred_flat = predictions.reshape(R, P * S, Cv).numpy()

    # ── Visualise predictions ────────────────────────────────────────────────
    plot_prediction_samples(pred_flat, pressure, coords, normalizer, cfg)

    # ── Export CSV ────────────────────────────────────────────────────────────
    print("\nExporting predictions to CSV …")
    export_predictions_to_csv(pred_flat, coords, normalizer, cfg)

    print("\n✔ Pipeline complete. Outputs saved to:", cfg.output_dir)
    return history


# Run
main(CONFIG)